# Spherical DeepKriging — toy example (Google Colab)

Minimal smoke test: MRTS-sphere basis + small TensorFlow DeepKriging network.

Repository: [STLABTW/spherical-deepkriging](https://github.com/STLABTW/spherical-deepkriging)

In [ ]:
# @title Install package (PyPI)
import shutil
import subprocess
import sys

# If PyPI falls back to sdist, the C++ extension needs CMake + a C++ compiler.
if shutil.which("cmake") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-qq", "-y", "cmake", "g++"], check=False)

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "spherical-deepkriging"])

In [ ]:
import numpy as np
import tensorflow as tf

from spherical_deepkriging.configs import DeepKrigingModelConfig
from spherical_deepkriging.models.deep_kriging import DeepKrigingTrainer
from spherical_deepkriging.basis_functions.mrts_sphere.sphere_cpp import mrts_sphere

In [ ]:
tf.keras.utils.set_random_seed(0)
rng = np.random.default_rng(0)

num_points = 40
k_order = 4
num_knots = 10
epochs = 5
batch_size = 8

In [ ]:
lat_deg = rng.uniform(-60, 60, size=num_points).astype(np.float32)
lon_deg = rng.uniform(-180, 180, size=num_points).astype(np.float32)
X_all = np.column_stack([lat_deg, lon_deg]).astype(np.float32)

lat_rad = np.deg2rad(lat_deg)
lon_rad = np.deg2rad(lon_deg)
y = (
    np.sin(lat_rad) * np.cos(lon_rad)
    + 0.2 * np.sin(2.0 * lat_rad) * np.sin(lon_rad)
).astype(np.float32)
y = (y + 0.01 * rng.normal(size=num_points).astype(np.float32)).astype(np.float32)
y = y[:, None]

knot_idx = rng.choice(num_points, size=num_knots, replace=False)
knot = X_all[knot_idx]
basis_named = mrts_sphere(knot=knot, k=k_order, X=X_all)
phi_all = np.asarray(basis_named["mrts"], dtype=np.float32)

idx = np.arange(num_points)
rng.shuffle(idx)
n_train = int(0.7 * num_points)
train_idx = idx[:n_train]
val_idx = idx[n_train:]

X_train = phi_all[train_idx]
y_train = y[train_idx]
X_val = phi_all[val_idx]
y_val = y[val_idx]

In [ ]:
def toy_mse_loss(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true - y_pred))


cfg = DeepKrigingModelConfig(
    input_dim=k_order,
    output_type="continuous",
    hidden_layers=[16, 16],
    dropout_rate=0.0,
    activation="relu",
    epochs=epochs,
    batch_size=batch_size,
    verbose=1,
    loss=toy_mse_loss,
    metrics=["mse", "mae"],
)

trainer = DeepKrigingTrainer(cfg)
history = trainer.train(
    train_features=X_train,
    train_labels=y_train,
    valid_features=X_val,
    valid_labels=y_val,
    log_dir=None,
)

In [ ]:
y_pred = trainer.model(X_val, training=False).numpy()
mse = np.mean((y_val - y_pred) ** 2)
mae = np.mean(np.abs(y_val - y_pred))

print("y_pred shape:", y_pred.shape)
print(f"val MSE: {mse:.6f}")
print(f"val MAE: {mae:.6f}")